# Boston Housing Regression Workflow


## Dataset
Included for upstream NNS example parity. The historical `b` variable has known ethical concerns.


In [1]:
from pathlib import Path
import csv

import numpy as np

from pynns import nns_dep, nns_m_reg, nns_part, nns_reg, nns_stack

np.set_printoptions(precision=4, suppress=True)

DATA_PATH = Path('docs/examples/notebooks/data/boston_housing.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('data/boston_housing.csv')


In [2]:
def load_boston_csv(path: Path) -> tuple[list[str], np.ndarray, np.ndarray]:
    with path.open(newline='') as handle:
        reader = csv.DictReader(handle)
        rows = list(reader)

    if not rows or reader.fieldnames is None:
        raise ValueError(f'No rows found in {path}')

    columns = list(reader.fieldnames)
    if columns[-1] != 'medv':
        raise ValueError('Expected medv to be the final target column')

    values = np.array(
        [[float(row[column]) for column in columns] for row in rows],
        dtype=np.float64,
    )
    return columns[:-1], values[:, :-1], values[:, -1]


def rmse(predicted: np.ndarray, actual: np.ndarray) -> float:
    predicted = np.asarray(predicted, dtype=np.float64)
    actual = np.asarray(actual, dtype=np.float64)
    return float(np.sqrt(np.mean((predicted - actual) ** 2)))


def mae(predicted: np.ndarray, actual: np.ndarray) -> float:
    predicted = np.asarray(predicted, dtype=np.float64)
    actual = np.asarray(actual, dtype=np.float64)
    return float(np.mean(np.abs(predicted - actual)))


def fit_linear(x_train: np.ndarray, y_train: np.ndarray, x_test: np.ndarray) -> np.ndarray:
    train_design = np.column_stack((np.ones(x_train.shape[0]), x_train))
    test_design = np.column_stack((np.ones(x_test.shape[0]), x_test))
    coefficients = np.linalg.lstsq(train_design, y_train, rcond=None)[0]
    return test_design @ coefficients


def take_columns(x: np.ndarray, names: list[str], wanted: tuple[str, ...]) -> np.ndarray:
    indices = [names.index(name) for name in wanted]
    return x[:, indices]


def print_table(headers: tuple[str, ...], rows: list[tuple[object, ...]]) -> None:
    rendered_rows = [[format_value(value) for value in row] for row in rows]
    widths = [len(header) for header in headers]
    for row in rendered_rows:
        widths = [max(width, len(value)) for width, value in zip(widths, row)]

    header_line = '  '.join(header.ljust(width) for header, width in zip(headers, widths))
    rule_line = '  '.join('-' * width for width in widths)
    print(header_line)
    print(rule_line)
    for row in rendered_rows:
        print('  '.join(value.ljust(width) for value, width in zip(row, widths)))


def format_value(value: object) -> str:
    if isinstance(value, (float, np.floating)):
        return f'{float(value):.4f}'
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    return str(value)


## Load data


In [3]:
feature_names, x, y = load_boston_csv(DATA_PATH)

print('data path:', DATA_PATH)
print('rows:', x.shape[0])
print('predictors:', x.shape[1])
print('target:', 'medv')
print('features:', ', '.join(feature_names))

preview_rows = []
for row_index in range(5):
    preview_rows.append(
        (
            row_index,
            x[row_index, feature_names.index('lstat')],
            x[row_index, feature_names.index('rm')],
            x[row_index, feature_names.index('nox')],
            y[row_index],
        )
    )

print_table(('row', 'lstat', 'rm', 'nox', 'medv'), preview_rows)


data path: docs/examples/notebooks/data/boston_housing.csv
rows: 506
predictors: 13
target: medv
features: crim, zn, indus, chas, nox, rm, age, dis, rad, tax, ptratio, b, lstat
row  lstat   rm      nox     medv   
---  ------  ------  ------  -------
0    4.9800  6.5750  0.5380  24.0000
1    9.1400  6.4210  0.4690  21.6000
2    4.0300  7.1850  0.4690  34.7000
3    2.9400  6.9980  0.4580  33.4000
4    5.3300  7.1470  0.4580  36.2000


In [4]:
summary_rows = []
for name, values in (
    ('medv', y),
    ('lstat', x[:, feature_names.index('lstat')]),
    ('rm', x[:, feature_names.index('rm')]),
    ('nox', x[:, feature_names.index('nox')]),
    ('ptratio', x[:, feature_names.index('ptratio')]),
):
    summary_rows.append(
        (
            name,
            float(np.min(values)),
            float(np.quantile(values, 0.25)),
            float(np.median(values)),
            float(np.quantile(values, 0.75)),
            float(np.max(values)),
        )
    )

print_table(('column', 'min', 'q25', 'median', 'q75', 'max'), summary_rows)


column   min      q25      median   q75      max    
-------  -------  -------  -------  -------  -------
medv     5.0000   17.0250  21.2000  25.0000  50.0000
lstat    1.7300   6.9500   11.3600  16.9550  37.9700
rm       3.5610   5.8855   6.2085   6.6235   8.7800 
nox      0.3850   0.4490   0.5380   0.6240   0.8710 
ptratio  12.6000  17.4000  19.0500  20.2000  22.0000


## Train/test split


In [5]:
def stratified_train_test_split(
    target: np.ndarray,
    *,
    train_fraction: float = 0.70,
    bins: int = 10,
    seed: int = 12345,
) -> tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    ordered = np.argsort(target + rng.normal(0.0, 1e-9, size=target.size))
    train_parts: list[np.ndarray] = []
    test_parts: list[np.ndarray] = []

    for bin_indices in np.array_split(ordered, bins):
        shuffled = bin_indices.copy()
        rng.shuffle(shuffled)
        cutoff = int(np.ceil(train_fraction * shuffled.size))
        train_parts.append(shuffled[:cutoff])
        test_parts.append(shuffled[cutoff:])

    train = np.concatenate(train_parts).astype(np.int64)
    test = np.concatenate(test_parts).astype(np.int64)
    rng.shuffle(train)
    rng.shuffle(test)
    return train, test


train_idx, test_idx = stratified_train_test_split(y)
x_train, x_test = x[train_idx], x[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print('train rows:', train_idx.size)
print('test rows:', test_idx.size)
print_table(
    ('split', 'mean medv', 'std medv', 'min medv', 'max medv'),
    [
        ('train', float(np.mean(y_train)), float(np.std(y_train)), float(np.min(y_train)), float(np.max(y_train))),
        ('test', float(np.mean(y_test)), float(np.std(y_test)), float(np.min(y_test)), float(np.max(y_test))),
    ],
)


train rows: 356
test rows: 150
split  mean medv  std medv  min medv  max medv
-----  ---------  --------  --------  --------
train  22.5907    9.3852    5.0000    50.0000 
test   22.3953    8.7005    5.6000    50.0000 


## Dependence scan


In [6]:
dependence_rows = []
for column_index, name in enumerate(feature_names):
    dep = nns_dep(x[:, column_index], y)
    pearson = float(np.corrcoef(x[:, column_index], y)[0, 1])
    dependence_rows.append(
        (
            name,
            float(dep['Correlation']),
            float(dep['Dependence']),
            pearson,
            abs(pearson),
        )
    )

ranked = sorted(dependence_rows, key=lambda row: max(row[2], row[4]), reverse=True)
print_table(('feature', 'NNS cor', 'NNS dep', 'Pearson', '|Pearson|'), ranked[:10])


feature  NNS cor  NNS dep  Pearson  |Pearson|
-------  -------  -------  -------  ---------
lstat    -0.2953  0.4846   -0.7377  0.7377   
rm       0.2525   0.5128   0.6954   0.6954   
dis      0.2739   0.5478   0.2499   0.2499   
b        0.2367   0.5369   0.3335   0.3335   
nox      -0.1276  0.5318   -0.4273  0.4273   
indus    0.1694   0.5244   -0.4837  0.4837   
crim     -0.0465  0.5213   -0.3883  0.3883   
ptratio  -0.1333  0.4583   -0.5078  0.5078   
tax      0.0447   0.4887   -0.4685  0.4685   
rad      0.0925   0.4866   -0.3816  0.3816   


## R-example stack path


In [7]:
full_linear_pred = fit_linear(x_train, y_train, x_test)
full_stack = nns_stack(
    x_train,
    y_train,
    x_test,
    obj_fn=rmse,
    objective='min',
    folds=3,
    cv_size=0.25,
    method=(1, 2),
    random_seed=12345,
)

full_rows = [
    ('linear least squares, all features', rmse(full_linear_pred, y_test), mae(full_linear_pred, y_test)),
    ('NNS.stack reg, all features', rmse(full_stack['reg'], y_test), mae(full_stack['reg'], y_test)),
    ('NNS.stack dim.red, all features', rmse(full_stack['dim.red'], y_test), mae(full_stack['dim.red'], y_test)),
    ('NNS.stack combined, all features', rmse(full_stack['stack'], y_test), mae(full_stack['stack'], y_test)),
]

print_table(('model', 'RMSE', 'MAE'), full_rows)
print('selected n.best:', full_stack['NNS.reg.n.best'])
print('selected dim.red threshold:', full_stack['NNS.dim.red.threshold'])


model                               RMSE    MAE   
----------------------------------  ------  ------
linear least squares, all features  4.7785  3.3324
NNS.stack reg, all features         5.7667  4.0573
NNS.stack dim.red, all features     5.6867  3.8826
NNS.stack combined, all features    5.6705  3.9355
selected n.best: 1.0
selected dim.red threshold: 0.68


## Focused multivariate stack


In [8]:
selected_features = ('lstat', 'rm', 'ptratio', 'tax', 'nox', 'dis')
x_selected = take_columns(x, feature_names, selected_features)
x_selected_train = x_selected[train_idx]
x_selected_test = x_selected[test_idx]

selected_linear_pred = fit_linear(x_selected_train, y_train, x_selected_test)
selected_nns = nns_stack(
    x_selected_train,
    y_train,
    x_selected_test,
    obj_fn=rmse,
    objective='min',
    folds=3,
    cv_size=0.25,
    method=(1,),
    random_seed=12345,
)

print('selected features:', ', '.join(selected_features))
print_table(
    ('model', 'RMSE', 'MAE'),
    [
        ('linear least squares, selected features', rmse(selected_linear_pred, y_test), mae(selected_linear_pred, y_test)),
        ('NNS direct multivariate, selected features', rmse(selected_nns['stack'], y_test), mae(selected_nns['stack'], y_test)),
    ],
)
print('selected n.best:', selected_nns['NNS.reg.n.best'])


selected features: lstat, rm, ptratio, tax, nox, dis
model                                       RMSE    MAE   
------------------------------------------  ------  ------
linear least squares, selected features     4.9348  3.4345
NNS direct multivariate, selected features  3.8150  2.7360
selected n.best: 1.0


In [9]:
comparison_rows = []
for row_number in range(10):
    comparison_rows.append(
        (
            row_number,
            y_test[row_number],
            selected_linear_pred[row_number],
            selected_nns['stack'][row_number],
            selected_nns['stack'][row_number] - y_test[row_number],
        )
    )

print_table(('row', 'actual', 'linear', 'NNS', 'NNS error'), comparison_rows)


row  actual   linear   NNS      NNS error
---  -------  -------  -------  ---------
0    8.8000   14.6498  11.0000  2.2000   
1    44.8000  39.6457  48.3000  3.5000   
2    20.5000  19.5585  18.8000  -1.7000  
3    14.9000  17.0042  16.1000  1.2000   
4    24.8000  25.7686  22.9000  -1.9000  
5    35.1000  34.7110  35.4000  0.3000   
6    13.1000  18.8582  12.5000  -0.6000  
7    19.9000  16.1999  19.0000  -0.9000  
8    37.0000  31.8112  30.5000  -6.5000  
9    18.5000  19.0314  19.5000  1.0000   


## Direct multivariate regression


In [10]:
n_best = int(round(float(selected_nns['NNS.reg.n.best'])))
direct_model = nns_m_reg(
    x_selected_train,
    y_train,
    point_est=x_selected_test[:5],
    n_best=n_best,
    confidence_interval=None,
)

direct_rows = []
for row_number, prediction in enumerate(direct_model['Point.est']):
    direct_rows.append((row_number, y_test[row_number], prediction))

print('training fitted R2:', round(float(direct_model['R2']), 4))
print('n_best used:', n_best)
print_table(('row', 'actual', 'nns_m_reg point estimate'), direct_rows)


training fitted R2: 0.9995
n_best used: 1
row  actual   nns_m_reg point estimate
---  -------  ------------------------
0    8.8000   11.0000                 
1    44.8000  48.3000                 
2    20.5000  18.8000                 
3    14.9000  16.1000                 
4    24.8000  22.9000                 


## Univariate view


In [11]:
lstat = x[:, feature_names.index('lstat')]
lstat_points = np.quantile(lstat, [0.10, 0.50, 0.90])
lstat_fit = nns_reg(
    lstat,
    y,
    point_est=lstat_points,
    order=3,
    confidence_interval=None,
    noise_reduction='median',
)
lstat_partitions = nns_part(
    lstat,
    y,
    order=3,
    obs_req=20,
    type='XONLY',
    noise_reduction='median',
)

print('lstat-only R2:', round(float(lstat_fit['R2']), 4))
print_table(
    ('lstat quantile point', 'estimated medv'),
    [(point, estimate) for point, estimate in zip(lstat_points, lstat_fit['Point.est'])],
)
print('partition order:', lstat_partitions['order'])
rp = lstat_partitions['regression.points']
print_table(
    ('partition x', 'partition y'),
    [(x_value, y_value) for x_value, y_value in zip(rp['x'][:8], rp['y'][:8])],
)


lstat-only R2: 0.6822
lstat quantile point  estimated medv
--------------------  --------------
4.6800                33.0533       
11.3600               21.6140       
23.0350               13.5108       
partition order: 3
partition x  partition y
-----------  -----------
5.0400       31.2000    
9.2350       22.8500    
14.0000      19.6000    
21.2300      13.8000    


## Classification path


In [12]:
high_value = np.where(y >= 25.0, 2.0, 1.0)
high_train = high_value[train_idx]
high_test = high_value[test_idx]

class_model = nns_stack(
    x_selected_train,
    high_train,
    x_selected_test,
    type='class',
    folds=3,
    cv_size=0.25,
    method=(1,),
    random_seed=12345,
)
class_pred = class_model['stack']
accuracy = float(np.mean(class_pred == high_test))
majority_accuracy = float(max(np.mean(high_test == 1.0), np.mean(high_test == 2.0)))

counts = []
for actual in (1.0, 2.0):
    for predicted in (1.0, 2.0):
        counts.append((int(actual), int(predicted), int(np.sum((high_test == actual) & (class_pred == predicted)))))

print('NNS classification accuracy:', round(accuracy, 4))
print('test-set majority-class accuracy:', round(majority_accuracy, 4))
print_table(('actual class', 'predicted class', 'count'), counts)
print('selected n.best:', class_model['NNS.reg.n.best'])


NNS classification accuracy: 0.88
test-set majority-class accuracy: 0.7467
actual class  predicted class  count
------------  ---------------  -----
1             1                101  
1             2                11   
2             1                7    
2             2                31   
selected n.best: 1.0


## Summary
